# Agentic Memory - Healthcare Scale Experiments (Colab web UI)

Web-UI variant of `agentic-memory-experiments.ipynb`. Same ingest,
same embedding, same schema, same tier ladder - minus the fallback
code paths that the Cursor / VSCode Colab extension needs.

If you opened this in the extension, stop and open the other notebook
instead. Some cells here assume `google.colab.userdata.get()` works,
which it doesn't from the extension.

See `experiments/healthcare/synthea-experiments-scale.md` for the full
Plan B design.

## What this notebook does

1. Mount Drive and stream Synthea FHIR tarballs from your MyDrive layout.
2. Bridge to the Neo4j experiment host over Tailscale userspace networking.
3. Load the Phase 0 `nemotron_local` embedding model on GPU.
4. Run the tier-selected ingest loop with patient-atomic checkpointing.
5. Verify Neo4j state with a handful of post-ingest sanity queries.


## Prerequisites (web UI)

Do these once, before the first run.

### 1. Phase 0 code changes merged
- [x] `fhir_loader.py` lazy tar iteration.
- [x] `bridge.py` UTF-8 subprocess decoding.
- [x] `EmbeddingService` supports `provider="nemotron_local"`.
- [x] `ConnectionManager.setup_database(embedding_dim=...)` reconciles vector indexes.

### 2. Neo4j host reachable over Tailscale
- Hetzner CPX52 running Neo4j 5.x.
- Tailscale on the host, joined to the same tailnet as this notebook.
- `bolt://` listening on the host's Tailscale IP.

### 3. Runtime
- Open this notebook at https://colab.research.google.com.
- **Runtime -> Change runtime type -> GPU** (T4 or better).
- Leave "Connect to" set to a hosted runtime.

### 4. Secrets (Colab Secrets sidebar)
Open Colab's **Secrets** sidebar (key icon, left gutter) and add each
key below, with "Notebook access" toggled ON:

| Key | Value |
|---|---|
| `TAILSCALE_AUTH_KEY` | ephemeral Tailscale pre-auth key, tag `tag:colab` |
| `NEO4J_URI` | `bolt://<tailscale-ip-or-hostname>:7687` |
| `NEO4J_USER` | `neo4j` |
| `NEO4J_PASSWORD` | (the experiment Neo4j password) |
| `STDB_URI` | SpacetimeDB Maincloud URL |
| `STDB_TOKEN` | SpacetimeDB auth token |
| `STDB_MODULE_NAME` | SpacetimeDB module name (`agentic-memory-temporal`) |
| `STDB_BINDINGS_MODULE` | `packages/am-temporal-kg/generated-bindings/index.ts` |

Missing required secrets produce a clear error, not a silent default.

### 5. Drive layout
```
My Drive/
  kubuntu/agentic-memory/
    big-healtcare-data/
      synthetic-data/
        synthea_2017_02_27.tar.gz         (2K/20K tiers)
        synthea_1m_fhir_3_0_May_24.tar.gz (200K tier)
    experiments/
      healthcare/
        checkpoints/   (written by this notebook)
        results/       (written by this notebook)
```


## Section 1 — Runtime sanity

Before we install anything heavy, confirm the Colab runtime is actually
what we asked for (GPU attached, Python version reasonable, Drive
mountable). Each of these failing has a different remediation, so we
check them separately and fail fast.

In [2]:
"""Runtime sanity check.

Confirms the Colab VM has:
- A supported Python version for the project (>=3.10).
- A CUDA-capable GPU reachable from PyTorch.
- The expected Linux tools available (apt, curl) for later cells.

Failures here should stop the notebook. There is no point continuing
to install 5 GB of dependencies if the runtime is misconfigured.
"""
import platform
import shutil
import subprocess
import sys


def _assert_python_version(min_major: int = 3, min_minor: int = 10) -> None:
    """Fail fast if the Colab runtime is on a Python older than the project supports."""
    major, minor = sys.version_info.major, sys.version_info.minor
    if (major, minor) < (min_major, min_minor):
        raise RuntimeError(
            f"Python {major}.{minor} is too old — project requires "
            f">= {min_major}.{min_minor}. Switch the Colab runtime."
        )
    print(f"Python {platform.python_version()} OK")


def _assert_gpu_available() -> None:
    """Fail fast if no CUDA GPU is attached to this Colab runtime.

    We want a GPU because the embedding model (Nemotron) runs ~100x faster
    on GPU than CPU for our batch sizes. Without it, a 200K-patient tier
    becomes unworkably slow.
    """
    try:
        output = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
            text=True,
            timeout=10,
        )
    except (FileNotFoundError, subprocess.SubprocessError) as exc:
        raise RuntimeError(
            "nvidia-smi not available — Colab runtime has no GPU. "
            "Runtime -> Change runtime type -> pick a GPU instance."
        ) from exc

    gpu_line = output.strip().splitlines()[0] if output.strip() else ""
    if not gpu_line:
        raise RuntimeError("GPU present but no device info returned; retry the runtime.")
    print(f"GPU: {gpu_line}")


def _assert_basic_tools() -> None:
    """Confirm the shell tools later cells assume are installed."""
    for tool in ("curl", "apt-get", "git"):
        if shutil.which(tool) is None:
            raise RuntimeError(f"Required tool '{tool}' is missing from PATH.")
    print("Shell tools: curl, apt-get, git OK")


_assert_python_version()
_assert_gpu_available()
_assert_basic_tools()
print("\nRuntime sanity: PASS")


Python 3.12.13 OK
GPU: NVIDIA A100-SXM4-80GB, 81920 MiB
Shell tools: curl, apt-get, git OK

Runtime sanity: PASS


## Section 2 - Mount Drive and clone the repo

Two separate operations, one cell each, so a failure in one doesn't
mask the other. Drive is mounted read-only-ish (we write checkpoints
and results but never touch the source tarballs). The repo is cloned
fresh from GitHub; uploaded-repo detection lives in the Cursor variant.


In [ ]:
"""Mount Google Drive and set experiment paths.

Web UI only - the extension-compatible variant of this cell lives in
`agentic-memory-experiments.ipynb`. No ImportError fallback, no
timeout wrapper; we expect a plain Colab runtime where `drive.mount()`
triggers an OAuth tab and returns.
"""
from pathlib import Path

from google.colab import drive

DRIVE_MOUNT = Path("/content/drive")
drive.mount(str(DRIVE_MOUNT))

SYNTHEA_DIR = DRIVE_MOUNT / "MyDrive/kubuntu/agentic-memory/big-healtcare-data/synthetic-data"
EXPERIMENT_DIR = DRIVE_MOUNT / "MyDrive/kubuntu/agentic-memory/experiments/healthcare"
CHECKPOINT_DIR = EXPERIMENT_DIR / "checkpoints"
RESULTS_DIR = EXPERIMENT_DIR / "results"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

for label, path in [
    ("Drive mount", DRIVE_MOUNT),
    ("Synthea source", SYNTHEA_DIR),
    ("Experiment dir", EXPERIMENT_DIR),
    ("Checkpoints", CHECKPOINT_DIR),
    ("Results", RESULTS_DIR),
]:
    marker = "OK" if path.exists() else "MISSING"
    print(f"[{marker}] {label}: {path}")


In [ ]:
"""Clone the agentic-memory repo into the Colab scratch filesystem.

Web UI only - the Cursor variant also handles a pre-uploaded repo.
Here we just clone from GitHub and record the HEAD commit for
reproducibility.

Set REPO_REF to a tag, branch, or commit SHA to pin a specific version.
Default `main` is fine for development; pin for published experiments.
"""
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/jarmen423/agentic-memory.git"
REPO_REF = "main"
REPO_DIR = Path("/content/agentic-memory")

if REPO_DIR.exists():
    print(f"Repo already cloned at {REPO_DIR}; pulling latest {REPO_REF}")
    subprocess.check_call(["git", "-C", str(REPO_DIR), "fetch", "--all", "--tags"])
    subprocess.check_call(["git", "-C", str(REPO_DIR), "checkout", REPO_REF])
    subprocess.check_call(["git", "-C", str(REPO_DIR), "pull", "--ff-only"])
else:
    subprocess.check_call(["git", "clone", REPO_URL, str(REPO_DIR)])
    subprocess.check_call(["git", "-C", str(REPO_DIR), "checkout", REPO_REF])

commit = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"], text=True
).strip()
print(f"Repo at commit {commit} ({REPO_REF})")

os.environ["PYTHONPATH"] = f"{REPO_DIR}/src:{os.environ.get('PYTHONPATH', '')}"
os.chdir(REPO_DIR)


In [ ]:
"""Install Python and Node dependencies.

Python:
    - Project in editable mode (/content/agentic-memory).
    - GPU embedding stack with pinned major versions so Colab's
      preinstalled torch is not clobbered.
Node (for the SpacetimeDB bridge):
    - `npm ci` (or `npm install`) in packages/am-temporal-kg so
      `npx tsx` can resolve tsx, typescript, and the spacetimedb client.
"""
import subprocess
import sys
from pathlib import Path

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "-e", "/content/agentic-memory",
])
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers>=4.44",
    "accelerate>=0.33",
    "sentencepiece>=0.2",
    "einops>=0.8",
])
print("Python dependencies installed.")

TS_PKG_DIR = Path("/content/agentic-memory/packages/am-temporal-kg")
npm_cmd = "ci" if (TS_PKG_DIR / "package-lock.json").exists() else "install"
print(f"Running `npm {npm_cmd}` in {TS_PKG_DIR}")
subprocess.check_call(
    ["npm", npm_cmd, "--no-audit", "--no-fund", "--silent"],
    cwd=str(TS_PKG_DIR),
)
print("Node dependencies installed (tsx, typescript, spacetimedb client).")


## Section 3 — Tailscale bridge

The Neo4j instance lives on a private Tailscale IP on our experiment VM.
Colab is on the public internet with no Tailscale client installed by
default, so we install Tailscale *inside* the Colab runtime in userspace
mode (no TUN device, no root kernel modules — just a SOCKS proxy and
a routing daemon).

This cell:

1. Downloads the Tailscale binary for Linux x86_64.
2. Starts `tailscaled --tun=userspace-networking` as a background process.
3. Runs `tailscale up` with the pre-authorised key from Colab Secrets.
4. Waits until the Colab node shows a Tailscale IPv4.

After this cell runs, the Neo4j host's Tailscale IP is reachable from
Python via `neo4j.GraphDatabase.driver(os.environ["NEO4J_URI"])`. No
kernel tweaks, no port forwards, no public Bolt exposure.

**Why userspace networking:** Colab runtimes do not permit creating TUN
devices. Tailscale's userspace mode works around this by routing all
Tailscale traffic through the daemon itself. The daemon exposes a local
SOCKS5 proxy we could use for other protocols, but the Neo4j bolt driver
connects directly via the userspace-netstack so no proxy config is needed
for our use case. Throughput is capped around ~100 Mbps which is well
above what we need for batched embedding writes.

In [ ]:
"""Install the Tailscale binary into the Colab runtime.

We pull the static Linux x86_64 release rather than the apt package
because Colab's apt sources don't include it by default and adding a
third-party repo just to get one binary is unnecessary churn.
"""
import subprocess
import urllib.request
from pathlib import Path

TAILSCALE_VERSION = "1.72.1"
TAILSCALE_TARBALL = f"tailscale_{TAILSCALE_VERSION}_amd64.tgz"
TAILSCALE_URL = f"https://pkgs.tailscale.com/stable/{TAILSCALE_TARBALL}"

WORKDIR = Path("/content/tailscale-bin")
WORKDIR.mkdir(parents=True, exist_ok=True)

tarball_path = WORKDIR / TAILSCALE_TARBALL
if not tarball_path.exists():
    print(f"Downloading {TAILSCALE_URL}")
    urllib.request.urlretrieve(TAILSCALE_URL, tarball_path)

subprocess.check_call(["tar", "-xzf", str(tarball_path), "-C", str(WORKDIR), "--strip-components=1"])

TAILSCALE_BIN = WORKDIR / "tailscale"
TAILSCALED_BIN = WORKDIR / "tailscaled"

for binary in (TAILSCALE_BIN, TAILSCALED_BIN):
    if not binary.exists():
        raise RuntimeError(f"Tailscale extraction failed; {binary} missing.")

print(f"Tailscale {TAILSCALE_VERSION} installed at {WORKDIR}")


In [ ]:
"""Start the Tailscale daemon in userspace mode and join the tailnet.

The daemon runs for the lifetime of the Colab session. If the session
restarts, this cell must be re-run - `tailscale up --authkey` with an
ephemeral key is idempotent and cheap.

Web UI only: reads every secret through `google.colab.userdata.get()`.
Missing secrets raise here rather than later so the failure mode is
obvious.
"""
import os
import subprocess
import time
from pathlib import Path

from google.colab import userdata


def _get_secret(name: str) -> str:
    value = userdata.get(name)
    if not value:
        raise RuntimeError(
            f"Secret {name!r} not set. Open Colab's Secrets sidebar "
            "(key icon) and add it with Notebook access ON."
        )
    os.environ[name] = value
    return value


AUTHKEY = _get_secret("TAILSCALE_AUTH_KEY")

STATE_DIR = Path("/content/tailscale-state")
STATE_DIR.mkdir(parents=True, exist_ok=True)
TAILSCALE_SOCK = STATE_DIR / "tailscaled.sock"

daemon_cmd = [
    str(TAILSCALED_BIN),
    "--tun=userspace-networking",
    f"--socket={TAILSCALE_SOCK}",
    f"--state={STATE_DIR / 'tailscaled.state'}",
    "--socks5-server=localhost:1055",
]

daemon_log = open(STATE_DIR / "tailscaled.log", "w", buffering=1)
daemon_proc = subprocess.Popen(
    daemon_cmd,
    stdout=daemon_log,
    stderr=subprocess.STDOUT,
    cwd=str(STATE_DIR),
)

time.sleep(2)
if daemon_proc.poll() is not None:
    raise RuntimeError(
        f"tailscaled exited immediately (code {daemon_proc.returncode}). "
        f"Check {STATE_DIR / 'tailscaled.log'}."
    )

up_cmd = [
    str(TAILSCALE_BIN),
    f"--socket={TAILSCALE_SOCK}",
    "up",
    f"--authkey={AUTHKEY}",
    "--hostname=colab-experiment-runner",
    "--accept-routes",
    "--ssh=false",
]
subprocess.check_call(up_cmd)

status_out = subprocess.check_output(
    [str(TAILSCALE_BIN), f"--socket={TAILSCALE_SOCK}", "ip", "-4"],
    text=True,
)
tailscale_ip = status_out.strip().splitlines()[0]
print(f"Colab node joined tailnet. Tailscale IPv4 = {tailscale_ip}")


## Section 4 — Configuration

One cell, one `RunConfig` dataclass, one source of truth for every
setting that downstream cells read. Putting all knobs in one place
means:

- The notebook's behaviour for a given tier is auditable at a glance.
- Re-running a subset of cells after tweaking a single value (e.g.
  batch size) doesn't require hunting through a dozen spots.
- The same config pattern can be serialised to a JSON alongside results
  for reproducibility.

**Pick the tier here.** Every other setting is derived.

In [ ]:
"""Build the RunConfig for this notebook invocation.

Tiers map to the scale-ladder from Plan B (synthea-experiments-scale.md):
    smoke  - 2,000 patients
    mid    - 20,000 patients
    scale  - 200,000 patients
    full   - 1,000,000 patients (research-credit VM only; not typical for Colab)

Change TIER below to pick a run. Everything else follows.
"""
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Literal
import json
import os

from google.colab import userdata

Tier = Literal["smoke", "mid", "scale", "full"]

TIER: Tier = "smoke"


@dataclass
class RunConfig:
    """Per-run settings for a single tier of the scale ladder.

    Attributes:
        tier: Which tier of the ladder we are running.
        max_patients: Upper bound on unique patients ingested from the tarball.
        checkpoint_every: How many patients between checkpoint writes.
        embed_batch_size: Records per GPU embedding batch.
        neo4j_write_batch_size: Records per Neo4j UNWIND batch.
        embedding_dim: Target embedding dimension for the vector index.
            Reconciled against the live embedder in the Section 6 cell.
        source_tarball: Path to the FHIR tarball on mounted Drive.
        project_id: SpacetimeDB project namespace; identifies this experiment's claims.
        checkpoint_path: Where to persist last-processed-patient state on Drive.
        results_path: Where to write the per-tier result JSON.
    """

    tier: Tier
    max_patients: int
    checkpoint_every: int
    embed_batch_size: int
    neo4j_write_batch_size: int
    embedding_dim: int
    source_tarball: Path
    project_id: str
    checkpoint_path: Path
    results_path: Path


_TIER_PRESETS: dict[Tier, dict] = {
    "smoke": {
        "max_patients": 2_000,
        "checkpoint_every": 250,
        "embed_batch_size": 64,
        "neo4j_write_batch_size": 500,
        "source_tarball_name": "synthea_2017_02_27.tar.gz",
    },
    "mid": {
        "max_patients": 20_000,
        "checkpoint_every": 1_000,
        "embed_batch_size": 128,
        "neo4j_write_batch_size": 1_000,
        "source_tarball_name": "synthea_2017_02_27.tar.gz",
    },
    "scale": {
        "max_patients": 200_000,
        "checkpoint_every": 2_000,
        "embed_batch_size": 128,
        "neo4j_write_batch_size": 1_000,
        "source_tarball_name": "synthea_1m_fhir_3_0_May_24.tar.gz",
    },
    "full": {
        "max_patients": 1_000_000,
        "checkpoint_every": 5_000,
        "embed_batch_size": 256,
        "neo4j_write_batch_size": 2_000,
        "source_tarball_name": "synthea_1m_fhir_3_0_May_24.tar.gz",
    },
}


def build_run_config(tier: Tier) -> RunConfig:
    preset = _TIER_PRESETS[tier]
    return RunConfig(
        tier=tier,
        max_patients=preset["max_patients"],
        checkpoint_every=preset["checkpoint_every"],
        embed_batch_size=preset["embed_batch_size"],
        neo4j_write_batch_size=preset["neo4j_write_batch_size"],
        # Matches the Phase 0 default nemotron_local model
        # (intfloat/multilingual-e5-large-instruct). Reconciled in the
        # embedding cell against the live embedder.
        embedding_dim=1024,
        source_tarball=SYNTHEA_DIR / preset["source_tarball_name"],
        project_id=f"synthea-scale-{tier}",
        checkpoint_path=CHECKPOINT_DIR / f"{tier}.json",
        results_path=RESULTS_DIR / f"ingest_{tier}.json",
    )


CONFIG = build_run_config(TIER)

# Required secrets for downstream cells. Each must be set in Colab
# Secrets before running this cell. Populate os.environ so the existing
# agentic_memory modules (Neo4j driver, SpacetimeDB bridge) see them.
_REQUIRED_SECRETS = (
    "NEO4J_URI",
    "NEO4J_USER",
    "NEO4J_PASSWORD",
    "STDB_URI",
    "STDB_TOKEN",
    "STDB_MODULE_NAME",
    "STDB_BINDINGS_MODULE",
)
_OPTIONAL_SECRETS = ("NVIDIA_API_KEY",)


def _load_secret(name: str, *, required: bool) -> str | None:
    value = userdata.get(name)
    if value:
        os.environ[name] = value
        return value
    if required:
        raise RuntimeError(
            f"Required secret {name!r} is not set in Colab Secrets. "
            "Open the Secrets sidebar (key icon) and add it."
        )
    return None


for secret_key in _REQUIRED_SECRETS:
    _load_secret(secret_key, required=True)

for secret_key in _OPTIONAL_SECRETS:
    _load_secret(secret_key, required=False)

os.environ["EMBEDDING_PROVIDER"] = "nemotron_local"
os.environ.setdefault("STDB_CONFIRMED_READS", "true")

print(json.dumps(
    {k: str(v) for k, v in asdict(CONFIG).items()},
    indent=2,
    default=str,
))

if not CONFIG.source_tarball.exists():
    raise RuntimeError(
        f"Source tarball not found on Drive: {CONFIG.source_tarball}\n"
        "Confirm Drive is mounted and the tarball path matches your layout."
    )


## Section 5 — Connectivity preflight

Long ingest runs fail expensively. Before we load any models or touch
any data, confirm we can reach every external dependency:

- **Neo4j** over Bolt on the Tailscale IP.
- **SpacetimeDB Maincloud** via the `TemporalBridge` RPC.
- **Drive tarball** readable (already checked above, but we also try
  cracking it open here to confirm the FHIR loader's lazy-iter path
  works).

Any one of these failing means we stop now, fix it on the host side,
and resume. We do not try to ingest partially.

In [ ]:
"""Preflight: Neo4j connectivity over Tailscale.

Runs the cheapest possible query that exercises the full client path:
driver handshake + authentication + one trivial Cypher roundtrip. If
this passes, the host is reachable and the credentials work.
"""
import os

from neo4j import GraphDatabase

driver = GraphDatabase.driver(
    os.environ["NEO4J_URI"],
    auth=(os.environ["NEO4J_USER"], os.environ["NEO4J_PASSWORD"]),
)

try:
    with driver.session() as session:
        result = session.run("RETURN 1 AS ok").single()
        assert result and result["ok"] == 1
    print(f"Neo4j OK — {os.environ['NEO4J_URI']}")
finally:
    driver.close()


In [ ]:
"""Preflight: SpacetimeDB Maincloud connectivity via TemporalBridge.

The bridge spawns a Node helper subprocess. We call is_available()
which internally performs a handshake — this surfaces the JSONDecodeError
we hit in preflight_checks.py if the Phase 0 fix isn't merged yet.

If this cell fails, check:
  1. STDB_BINDINGS_MODULE is an absolute path that exists.
  2. The Phase 0 bridge fix is merged (lines near _ensure_process).
  3. Maincloud is reachable at STDB_URI.
"""
from agentic_memory.temporal.bridge import TemporalBridge

_bridge = TemporalBridge.from_env()

if not _bridge.is_available():
    raise RuntimeError(
        f"SpacetimeDB bridge unavailable: {_bridge.disabled_reason}. "
        "Confirm STDB_URI/STDB_TOKEN/STDB_BINDINGS_MODULE secrets are set "
        "and Phase 0 bridge fixes are merged."
    )

print("SpacetimeDB bridge OK")


In [ ]:
"""Preflight: FHIR loader can open the tarball and yield the first patient.

This is the cheapest way to confirm:
  1. Drive is still mounted (Colab sometimes silently unmounts).
  2. The Phase 0 lazy-iter patch to fhir_loader.py is in effect —
     the unpatched loader calls outer.getmembers() which takes 15+ min
     on a 30 GB Drive-streamed archive. If this preflight takes more
     than ~60 seconds the patch is NOT applied.
  3. Bundles actually parse as FHIR R4.

We only read one patient. Full streaming happens in the ingest section.
"""
import time

from agentic_memory.healthcare.fhir_loader import SyntheaFHIRLoader

# Cap at 1 patient purely to measure "first record" latency without pulling
# the whole tarball into memory. Phase 0 lazy-iter should make this cheap.
_loader = SyntheaFHIRLoader(str(CONFIG.source_tarball), max_patients=1)

start = time.monotonic()
first_record = next(iter(_loader.iter_records()), None)
elapsed = time.monotonic() - start

if first_record is None:
    raise RuntimeError(
        f"FHIR loader returned zero records from {CONFIG.source_tarball}."
    )

print(f"First record yielded in {elapsed:.1f}s")
if elapsed > 60:
    print(
        "WARNING: first-record latency > 60s strongly suggests the Phase 0 "
        "lazy-iter patch is not applied to fhir_loader.py. Long runs will be "
        "much slower than necessary."
    )

print(f"  record_type = {first_record.get('record_type', '<missing>')}")
print(f"  fields present = {sorted(first_record.keys())[:8]}")



## Section 6 — Embedding model

Load the Nemotron embedding model onto the GPU **once** per session and
reuse it for every batch. The existing `EmbeddingService` dispatches on
`provider="nemotron"` — Phase 0 adds a local-GPU backend so the service
interface is identical whether you point it at NVIDIA's API or at a
local `transformers` model.

This cell is heavy (multi-GB model download on first run) and should
only be run once per Colab session. If you re-run later sections and
see "model not loaded" errors, re-run this cell.

In [ ]:
"""Build the Nemotron-backed EmbeddingService and warm the GPU.

Uses the Phase 0 ``nemotron_local`` provider, which loads a HuggingFace
text encoder (default: ``intfloat/multilingual-e5-large-instruct``,
1024-dim) onto CUDA and wraps it in the same interface OpenAI/Gemini
providers use.

Override the default model via the ``AM_LOCAL_EMBED_MODEL`` env var if
we switch to an NVIDIA Nemotron retriever checkpoint later. The output
dimension is auto-detected from a warm-up probe, so the Neo4j vector
index dimension (see schema-setup cell) is reconciled to whatever the
loaded model actually produces — no manual dim arithmetic required.

Why this cell runs before schema setup:
    ``ConnectionManager.setup_database(embedding_dim=...)`` needs the
    final dimension at DDL time. Instantiating the embedder here lets
    us read ``embedder.dimensions`` off the live object and push it
    into CONFIG so every downstream cell agrees on the number.
"""
import os

from agentic_memory.core.embedding import EmbeddingService

embedder = EmbeddingService(
    provider="nemotron_local",
    # Both are ignored by the local-GPU backend but harmless to pass. They
    # matter only if we fall back to the hosted Nemotron API provider.
    api_key=os.environ.get("NVIDIA_API_KEY"),
)

# Warm the GPU by embedding a trivial batch. First call typically takes
# 30-90 s because the model weights load into VRAM here. Subsequent
# calls are fast (see Section 9 ingest loop for throughput numbers).
_warm_start_texts = [
    "Condition: Hypertensive disorder, systemic arterial (disorder)",
    "Medication: Lisinopril 10 MG Oral Tablet",
    "Observation: Systolic blood pressure 142 mmHg",
]
_warm_vectors = embedder.embed_batch(_warm_start_texts)

if not _warm_vectors:
    raise RuntimeError("Warm-start embed_batch() returned no vectors.")

detected_dim = len(_warm_vectors[0])

# Reconcile RunConfig.embedding_dim with reality. If the preset was
# wrong (e.g. older notebook runs still claiming 2048), overwrite it
# so setup_database() below creates a matching vector index.
if CONFIG.embedding_dim != detected_dim:
    print(
        f"Embedding dim mismatch: CONFIG was {CONFIG.embedding_dim}, "
        f"model produced {detected_dim}. Updating CONFIG to {detected_dim}."
    )
    CONFIG.embedding_dim = detected_dim

print(f"Nemotron local-GPU embedding service ready. dim={detected_dim}")
print(f"Warm-start produced {len(_warm_vectors)} vectors.")


## Section 7 — Schema setup

Bootstrap the Neo4j schema: unique-ID constraints, entity constraints,
and the `healthcare_embeddings` vector index at our target dimension.

`ConnectionManager.setup_database()` is idempotent. Running it on an
already-set-up database is a no-op, which is what we want when
re-running a partially-ingested tier.

**Dimension mismatch is the one thing this will NOT auto-fix.** If the
index already exists at 3072-dim (the old Gemini default) and CONFIG
says 2048-dim, Neo4j refuses to silently change the dimension. The cell
detects this and fails loudly with a remediation hint; we then drop the
index manually on the Neo4j host and re-run this cell.

In [ ]:
"""Run setup_database() against the experiment Neo4j instance.

The ConnectionManager class manages the Neo4j driver and exposes schema
setup through setup_database(). We pass the target embedding dimension
so the vector index is created at 2048-dim for Nemotron.

Phase 0 extends setup_database() to accept an `embedding_dim` argument.
Until that lands, the default stays at 3072-dim — we detect that
mismatch here rather than discovering it mid-ingest when vector writes
silently truncate.
"""
from agentic_memory.core.connection import ConnectionManager

connection = ConnectionManager(
    os.environ["NEO4J_URI"],
    os.environ["NEO4J_USER"],
    os.environ["NEO4J_PASSWORD"],
)

try:
    connection.setup_database(embedding_dim=CONFIG.embedding_dim)
except TypeError as exc:
    raise RuntimeError(
        "ConnectionManager.setup_database() does not yet accept "
        "embedding_dim — Phase 0 schema change not merged. "
        "Abort this run and land that change first."
    ) from exc

# Verify the index actually came up at the expected dimension.
with connection.driver.session() as session:
    row = session.run(
        """
        SHOW VECTOR INDEXES
        YIELD name, options
        WHERE name = 'healthcare_embeddings'
        RETURN options.indexConfig.`vector.dimensions` AS dim
        """
    ).single()

if row is None:
    raise RuntimeError("healthcare_embeddings index did not come up after setup_database().")

actual_dim = int(row["dim"])
if actual_dim != CONFIG.embedding_dim:
    raise RuntimeError(
        f"healthcare_embeddings index is at {actual_dim}-dim but CONFIG expects "
        f"{CONFIG.embedding_dim}-dim. Drop the index on the host "
        "(DROP INDEX healthcare_embeddings) and re-run this cell."
    )

print(f"Schema OK. healthcare_embeddings vector index at {actual_dim}-dim.")


## Section 8 — Checkpoint state

Colab sessions die. 24-hour maximum, shorter on idle, faster on flaky
runtimes. A tier that takes 8 hours to ingest cannot afford to start
over every time.

The checkpoint design:

- **Unit of progress is the patient.** We commit a checkpoint after
  every N patients (N = `CONFIG.checkpoint_every`). Checkpoints do NOT
  commit mid-patient — a patient is atomic: all of their
  encounters/conditions/medications/observations/procedures go in
  together, or they don't.
- **Checkpoint state** is a JSON file on Drive:
  `{"tier": "...", "last_patient_id": "...", "patients_done": N, "started_at": "..."}`.
- **Resume** reads the checkpoint, skips any patient whose ID has
  already been processed (we track a set of patient IDs in-session),
  and continues. Neo4j writes are idempotent by deterministic MERGE
  keys so accidentally re-writing a record is a no-op.
- **Drive is the source of truth.** We do NOT rely on Neo4j "last
  patient seen" queries because they'd be slow and Neo4j state lives on
  a different host than the notebook.

In [ ]:
"""Checkpoint helpers — load and save per-tier progress state on Drive.

The helpers are tiny on purpose. The heavy work is the ingest loop;
checkpoint state is just enough to answer "where do I resume?"
"""
import json
from dataclasses import dataclass, asdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Optional


@dataclass
class CheckpointState:
    """Resumable ingest progress for a single tier.

    Attributes:
        tier: Tier name (matches RunConfig.tier).
        patients_done: Count of patients fully ingested so far.
        last_patient_id: ID of the most recent patient we committed.
            Used for human-readable logs; the skip logic uses the
            processed-IDs set, not this.
        processed_ids: Set of all patient IDs already ingested. Persisted
            so a resumed session can skip them.
        started_at: ISO timestamp of the first session for this tier.
        updated_at: ISO timestamp of the last checkpoint write.
    """

    tier: str
    patients_done: int
    last_patient_id: Optional[str]
    processed_ids: list[str]
    started_at: str
    updated_at: str


def load_checkpoint(path: Path, tier: str) -> CheckpointState:
    """Return the existing checkpoint for this tier, or a fresh empty one.

    Args:
        path: Path to the checkpoint JSON file on Drive.
        tier: Current tier name — used to guard against accidentally
            loading a different tier's checkpoint.

    Returns:
        Loaded CheckpointState, or a new empty one if no file exists.
    """
    if not path.exists():
        now = datetime.now(timezone.utc).isoformat()
        return CheckpointState(
            tier=tier,
            patients_done=0,
            last_patient_id=None,
            processed_ids=[],
            started_at=now,
            updated_at=now,
        )

    raw = json.loads(path.read_text())
    if raw.get("tier") != tier:
        raise RuntimeError(
            f"Checkpoint at {path} is for tier '{raw.get('tier')}', "
            f"not '{tier}'. Move it aside or run the matching tier."
        )

    return CheckpointState(**raw)


def save_checkpoint(path: Path, state: CheckpointState) -> None:
    """Atomically persist a checkpoint to Drive.

    Uses write-to-tmp-then-rename so a Colab crash during the write
    cannot leave a truncated file. Drive's rename is atomic from the
    mounted filesystem's perspective.

    Args:
        path: Target checkpoint file.
        state: Current in-memory checkpoint to persist.
    """
    state.updated_at = datetime.now(timezone.utc).isoformat()
    tmp_path = path.with_suffix(".json.tmp")
    tmp_path.write_text(json.dumps(asdict(state), indent=2))
    tmp_path.replace(path)


checkpoint_state = load_checkpoint(CONFIG.checkpoint_path, CONFIG.tier)
print(
    f"Checkpoint loaded for tier '{CONFIG.tier}': "
    f"{checkpoint_state.patients_done} patients done, "
    f"last = {checkpoint_state.last_patient_id}"
)


## Section 9 — Ingest loop

The main event. Reads FHIR records from the source tarball, routes each
through `HealthcareIngestionPipeline.ingest()`, which handles:

- Generating the memory-node embedding via our Nemotron GPU service.
- Writing the `:Memory:Healthcare:*` node to Neo4j (idempotent MERGE).
- Creating entity nodes and edges (patient, provider, diagnosis, etc.)
- Shadow-writing a temporal claim to SpacetimeDB when the record has a
  timestamp.

**Patient-atomic checkpointing.** We don't checkpoint between records
within a patient — only between patients. The FHIR loader already yields
records grouped by patient, so we detect patient boundaries by watching
the `PATIENT` field change and checkpoint on boundary crossings once
`CONFIG.checkpoint_every` boundaries have accumulated.

**Resume behaviour.** On session restart, the checkpoint gives us the
set of already-processed patient IDs. We stream the tarball from the
top (no way to seek into a gzip reliably) but skip any record whose
patient is already in that set. The skip is cheap — no embedding, no
Neo4j write, just a dict lookup.

**Latency expectations.**

- Smoke (2K): ~20–40 minutes total.
- Mid (20K): ~2–4 hours.
- Scale (200K): ~8–16 hours across 1–2 Colab sessions.

The dominant cost is GPU embedding time. Neo4j writes over Tailscale are
negligible in comparison once batched.

In [ ]:
"""Construct the ingestion pipeline for this run.

Mirrors scripts/ingest_synthea.py's main(): builds the pipeline with
the same services, wired to the Colab-side embedder and the remote
Neo4j instance.

enable_llm_extraction is hard-coded to False here. Structured
entity extraction from FHIR fields is free and deterministic; LLM
extraction is expensive and would cost ~$100+ at the scale tier. If
we ever need LLM extraction for a validation subset we do it from the
CLI script, not Colab.
"""
from agentic_memory.healthcare.pipeline import HealthcareIngestionPipeline

pipeline = HealthcareIngestionPipeline(
    connection_manager=connection,
    embedding_service=embedder,
    entity_extractor=None,
    temporal_bridge=_bridge,
    project_id=CONFIG.project_id,
    enable_llm_extraction=False,
)

print(f"Pipeline ready for project_id='{CONFIG.project_id}'")


In [ ]:
"""Run the ingest loop with patient-atomic checkpointing.

Outer loop: stream FHIR records; group by patient (via the PATIENT
field the FHIR loader puts on every row); ingest each record via the
pipeline; checkpoint after every Nth patient.

Skip-already-processed: on a resumed session, checkpoint_state holds
the set of patient IDs previously committed. Records whose patient is
already in the set are skipped without embedding or Neo4j work.

Stop conditions:
  - max_patients reached (tier-specific cap)
  - FHIR iterator exhausted (reached end of tarball)
  - KeyboardInterrupt (user-initiated stop; checkpoint is flushed)

The loop is long-running (hours on scale tier). We intentionally avoid
any inner-loop tqdm or rich formatting because Colab's output buffer
struggles with dense progress updates over long runs; plain timestamped
prints every batch are more robust.
"""
import logging
import time
from datetime import datetime, timezone

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
ingest_logger = logging.getLogger("ingest")

processed_id_set: set[str] = set(checkpoint_state.processed_ids)
current_patient_id: str | None = None
current_patient_records: list[dict] = []

records_seen = 0
records_written = 0
records_errored = 0
temporal_writes = 0
patients_since_checkpoint = 0

run_start = time.monotonic()

loader = SyntheaFHIRLoader(
    str(CONFIG.source_tarball),
    max_patients=CONFIG.max_patients,
)


def _flush_patient(patient_id: str, patient_records: list[dict]) -> tuple[int, int, int]:
    """Ingest all records for one patient through the pipeline.

    We flush a patient as a unit so the checkpoint can treat patients
    as atomic: either every row landed, or we re-do the patient from
    scratch on resume (idempotent MERGEs make this safe).

    Returns (written, errored, temporal_writes).
    """
    written = errored = temporal = 0
    for row in patient_records:
        try:
            result = pipeline.ingest(row)
            written += 1
            if result.get("temporal_written"):
                temporal += 1
        except Exception as exc:  # noqa: BLE001 — keep ingesting on row failure
            errored += 1
            ingest_logger.warning(
                "Failed ingest for patient=%s record_type=%s: %s",
                patient_id, row.get("record_type"), exc,
            )
    return written, errored, temporal


try:
    for record in loader.iter_records():
        records_seen += 1
        patient_id = record.get("PATIENT") or record.get("patient_id") or ""

        # Skip records for patients we already ingested in a prior session.
        if patient_id and patient_id in processed_id_set:
            continue

        # Patient boundary: the loader yields all records for one patient
        # contiguously, so a change in PATIENT means the previous patient
        # is complete.
        if patient_id != current_patient_id:
            if current_patient_id is not None and current_patient_records:
                w, e, t = _flush_patient(current_patient_id, current_patient_records)
                records_written += w
                records_errored += e
                temporal_writes += t

                processed_id_set.add(current_patient_id)
                checkpoint_state.patients_done += 1
                checkpoint_state.last_patient_id = current_patient_id
                patients_since_checkpoint += 1

                if patients_since_checkpoint >= CONFIG.checkpoint_every:
                    checkpoint_state.processed_ids = sorted(processed_id_set)
                    save_checkpoint(CONFIG.checkpoint_path, checkpoint_state)
                    patients_since_checkpoint = 0
                    elapsed_min = (time.monotonic() - run_start) / 60
                    ingest_logger.info(
                        "CHECKPOINT done=%d records_written=%d errors=%d temporal=%d elapsed_min=%.1f",
                        checkpoint_state.patients_done,
                        records_written,
                        records_errored,
                        temporal_writes,
                        elapsed_min,
                    )

            current_patient_id = patient_id
            current_patient_records = []

        current_patient_records.append(record)

    # End-of-stream: flush the final in-progress patient, if any.
    if current_patient_id is not None and current_patient_records:
        w, e, t = _flush_patient(current_patient_id, current_patient_records)
        records_written += w
        records_errored += e
        temporal_writes += t
        processed_id_set.add(current_patient_id)
        checkpoint_state.patients_done += 1
        checkpoint_state.last_patient_id = current_patient_id

except KeyboardInterrupt:
    ingest_logger.warning("User interrupt — flushing final checkpoint before exit.")

finally:
    # Always persist the latest state so partial runs are resumable.
    checkpoint_state.processed_ids = sorted(processed_id_set)
    save_checkpoint(CONFIG.checkpoint_path, checkpoint_state)

total_min = (time.monotonic() - run_start) / 60
print(
    f"\nIngest complete. patients_done={checkpoint_state.patients_done} "
    f"records_seen={records_seen} records_written={records_written} "
    f"errors={records_errored} temporal_writes={temporal_writes} "
    f"elapsed_min={total_min:.1f}"
)


## Section 10 — Post-ingest verification

After the ingest loop completes, run a handful of sanity queries
directly against Neo4j. These are not the real experiments — they just
catch "ingest silently wrote garbage" failure modes before we invest
hours in evaluation runs.

The checks:

- **Node counts by label.** If Condition nodes are zero but Encounter
  nodes are high, something is wrong with the conditions path.
- **Entity coverage.** Random Patient entities should have outgoing
  `HAD_ENCOUNTER`, `DIAGNOSED_WITH`, etc. If any patient has zero
  relationships, we dropped their records on the floor.
- **Vector index population.** At least one node should have a valid
  embedding of the expected dimension.
- **Temporal claim echo.** SpacetimeDB should report the same ballpark
  of claim counts as Neo4j temporal writes.

All four failing means something major is wrong. Any one passing while
the others fail usually points at one specific code path to debug.

In [ ]:
"""Run the post-ingest sanity checks and write a results JSON.

Every check prints a one-line result. The full payload is also saved
to CONFIG.results_path so we have a per-tier artifact alongside the
scale curve we eventually plot.
"""
import json
import random
from dataclasses import asdict

checks: dict[str, object] = {
    "tier": CONFIG.tier,
    "patients_done": checkpoint_state.patients_done,
}

with connection.driver.session() as session:
    # Node counts by healthcare record type.
    counts = {}
    for label in ("Encounter", "Condition", "Medication", "Observation", "Procedure"):
        row = session.run(
            f"MATCH (n:Memory:Healthcare:{label}) RETURN count(n) AS c"
        ).single()
        counts[label] = int(row["c"]) if row else 0
    checks["node_counts"] = counts

    # Entity counts.
    entity_counts = {}
    for label in ("Patient", "Provider", "Diagnosis", "Medication", "Procedure"):
        row = session.run(
            f"MATCH (n:Entity:{label}) RETURN count(n) AS c"
        ).single()
        entity_counts[label] = int(row["c"]) if row else 0
    checks["entity_counts"] = entity_counts

    # Random patient coverage — sample 5 patients, confirm each has >=1 relationship.
    sampled_coverage = []
    patients = session.run(
        "MATCH (p:Entity:Patient) RETURN p.name AS pid LIMIT 200"
    ).data()
    for entry in random.sample(patients, min(5, len(patients))):
        pid = entry["pid"]
        rels = session.run(
            "MATCH (:Entity:Patient {name: $pid})-[r]-() RETURN count(r) AS c",
            pid=pid,
        ).single()
        sampled_coverage.append({"patient": pid, "relationships": int(rels["c"]) if rels else 0})
    checks["sampled_patient_coverage"] = sampled_coverage

    # Vector index population sanity.
    vec_row = session.run(
        """
        MATCH (n:Memory:Healthcare)
        WHERE n.embedding IS NOT NULL
        RETURN count(n) AS c, size(head(collect(n.embedding))) AS dim
        """
    ).single()
    checks["embedded_nodes"] = int(vec_row["c"]) if vec_row else 0
    checks["embedding_dim_seen"] = int(vec_row["dim"]) if vec_row and vec_row["dim"] else None

# Write results.
CONFIG.results_path.parent.mkdir(parents=True, exist_ok=True)
CONFIG.results_path.write_text(json.dumps(checks, indent=2, default=str))

print(json.dumps(checks, indent=2, default=str))
print(f"\nResults written to {CONFIG.results_path}")


## Next steps

Once the ingest cell reports `patients_done == CONFIG.max_patients`
and post-ingest checks look healthy, the tier is complete. What
happens next:

1. **Keep Neo4j running.** Evaluation runs (Experiments 1 and 2)
   connect to the same Neo4j instance. You can run eval from this
   notebook if you want GPU-backed query-embedding, but CPU-side
   eval works fine too — the queries are cheap and the QA tasks
   are small.

2. **Move to the next tier.** Change `TIER` in the config cell
   (section 4) to `mid` or `scale`, then re-run from section 4
   onward. Sections 1–3 (runtime, Drive, Tailscale) don't need to
   rerun unless the Colab session died.

3. **Tear down.** When all tiers are done:
   - Download the final checkpoints and results JSONs from Drive.
   - `neo4j-admin database dump` on the host for an archive.
   - If using a rented VM, stop or delete it.
   - Revoke the Tailscale ephemeral authkey so Colab loses tailnet
     access.

Where findings live:

- Per-tier sanity JSON: `CONFIG.results_path` (on Drive).
- Experiment 1 (temporal decay) results: produced by the separate
  eval runner, not this notebook.
- Experiment 2 (multi-hop reasoning) results: ditto.
- Final scale-sensitivity curve: `docs/research/scale-validity-findings.md`
  referenced in Plan B.